# Fraud Detection Model - Demo Notebook
StablePay AI Engine
XGBoost-based fraud detection for merchant transactions

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb

print('Libraries loaded successfully')

In [ ]:
# Generate synthetic transaction data
np.random.seed(42)
n_samples = 10000

data = pd.DataFrame({
    'amount': np.random.exponential(1000, n_samples),
    'velocity_1h': np.random.poisson(2, n_samples),
    'velocity_24h': np.random.poisson(10, n_samples),
    'avg_amount_30d': np.random.exponential(900, n_samples),
    'is_cross_border': np.random.binomial(1, 0.2, n_samples),
    'hour_of_day': np.random.randint(0, 24, n_samples),
    'previous_chargebacks': np.random.poisson(0.1, n_samples),
})

# Create synthetic target (5% fraud rate)
data['is_fraud'] = (
    (data['amount'] > 5000) |
    (data['velocity_1h'] > 10) |
    (data['previous_chargebacks'] > 3)
).astype(int) * np.random.binomial(1, 0.3, n_samples)

data['is_fraud'] = np.clip(data['is_fraud'], 0, 1)
print(f'Fraud rate: {data["is_fraud"].mean():.2%}')
print(f'Dataset shape: {data.shape}')

In [ ]:
# Train XGBoost model
X = data.drop('is_fraud', axis=1)
y = data['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('\nClassification Report:')
print(classification_report(y_test, y_pred))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature Importance:')
print(importance) 